In [0]:

from urllib.request import urlretrieve
from pathlib import Path
from zipfile import ZipFile
import pandas as pd
%pip install openpyxl

In [0]:
%sql

-- 1. Catálogo exclusivo do BairroIQ
CREATE CATALOG IF NOT EXISTS bairroiq;

-- 2. Schemas das camadas
CREATE SCHEMA IF NOT EXISTS bairroiq.raw;

-- 3. Volume para os arquivos originais do IBGE
CREATE VOLUME IF NOT EXISTS bairroiq.raw.ibge_raw
COMMENT 'Arquivos originais do IBGE utilizados pelo projeto BairroIQ';

CREATE VOLUME IF NOT EXISTS bairroiq.raw.ssp_raw;

CREATE VOLUME IF NOT EXISTS bairroiq.raw.obsampa_raw;

In [0]:

BASE = Path("/Volumes/bairroiq/raw")

arquivos = {
    "ibge_raw": {
        "agregados_basicos.zip":
            "https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios/Agregados_por_Distrito_csv/Agregados_por_distritos_basico_BR_20260520.zip",
    },
    "ssp_raw": {
        "SPDadosCriminais_2022.xlsx":
            "https://www.ssp.sp.gov.br/assets/estatistica/transparencia/spDados/SPDadosCriminais_2022.xlsx",
        "DadosProdutividade_2024.xlsx":
            "https://www.ssp.sp.gov.br/assets/estatistica/transparencia/spDados/DadosProdutividade_2024.xlsx",
        "MDIP_2026.xlsx":
            "https://www.ssp.sp.gov.br/assets/estatistica/transparencia/spDados/MDIP_2026.xlsx",
        "CelularesSubtraidos_2022.xlsx":
            "https://www.ssp.sp.gov.br/assets/estatistica/transparencia/baseDados/celularesSub/CelularesSubtraidos_2022.xlsx",
        "VeiculosSubtraidos_2022.xlsx":
            "https://www.ssp.sp.gov.br/assets/estatistica/transparencia/baseDados/veiculosSub/VeiculosSubtraidos_2022.xlsx",
        "ObjetosSubtraidos_2022.xlsx":
            "https://www.ssp.sp.gov.br/assets/estatistica/transparencia/baseDados/objetosSub/ObjetosSubtraidos_2022.xlsx",
    },
    "obsampa_raw": {
        "ObservaSampaDadosAbertosIndicadoresCSV.csv":
            "https://dados-abertos-observasampa.prefeitura.sp.gov.br/_temp/DadosAbertos/ObservaSampaDadosAbertosIndicadoresCSV.csv",
    },
}

for volume, bases in arquivos.items():
    pasta = BASE / volume
    pasta.mkdir(parents=True, exist_ok=True)

    for nome, url in bases.items():
        destino = pasta / nome

        if destino.exists():
            print(f"Já existe, ignorando: {nome}")
            continue

        urlretrieve(url, destino)
        print(f"Baixado: {destino}")

print("Ingestão Raw concluída!")

In [0]:


BASE = Path("/Volumes/bairroiq/raw")

# 1. Inventário dos arquivos
arquivos = [p for p in BASE.rglob("*") if p.is_file()]

print(f"Total de arquivos encontrados: {len(arquivos)}")

for arquivo in sorted(arquivos):
    tamanho_mb = arquivo.stat().st_size / (1024 * 1024)
    print(f"{arquivo.name} | {tamanho_mb:.2f} MB")

# 2. Inspeção dos arquivos
for arquivo in sorted(arquivos):
    print(f"\n{'=' * 70}")
    print(f"ARQUIVO: {arquivo.name}")

    try:
        if arquivo.suffix.lower() == ".zip":
            with ZipFile(arquivo, "r") as zip_ref:
                for item in zip_ref.infolist():
                    print(f"  {item.filename} | {item.file_size / 1024:.1f} KB")

        elif arquivo.suffix.lower() == ".xlsx":
            excel = pd.ExcelFile(arquivo)
            print(f"Abas: {excel.sheet_names}")

            for aba in excel.sheet_names:
                amostra = pd.read_excel(
                    arquivo, sheet_name=aba, nrows=5
                )
                print(f"\nAba: {aba}")
                print(f"Colunas: {amostra.columns.tolist()}")
                display(amostra.head(3))

        elif arquivo.suffix.lower() == ".csv":
            amostra = pd.read_csv(
                arquivo, sep=None, engine="python", nrows=5
            )
            print(f"Colunas: {amostra.columns.tolist()}")
            display(amostra.head(3))

    except Exception as erro:
        print(f"Não foi possível inspecionar: {erro}")
